# Module 18 — Multimodal Local Applications

Companion notebook to [`docs/modules/18_multimodal_local_applications.md`](../docs/modules/18_multimodal_local_applications.md).

Real PDF rendering, layout/table extraction, and image preprocessing (`PyMuPDF`, `pdfplumber`,
`Pillow` — real libraries, not LLM runtimes) over two real sample PDFs: `sample_invoice.pdf`
(digital-native, real text and table) and `scanned_receipt.pdf` (a real image-only page with no
text layer at all). `should_use_vlm()` routes each one to the correct path automatically. Only
the VLM call itself is `FakeVLM`-backed.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_18"))
print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/bhakti/workspace/local_ai_app


## 1. The two real sample fixtures

In [2]:
from local_ai_core.multimodal.pdf_extraction import extract_text_layer

MULTIMODAL_DIR = REPO_ROOT / "datasets" / "multimodal"
invoice_text = extract_text_layer(MULTIMODAL_DIR / "sample_invoice.pdf", 0)
receipt_text = extract_text_layer(MULTIMODAL_DIR / "scanned_receipt.pdf", 0)

print(f"sample_invoice.pdf text layer: {len(invoice_text)} chars")
print(f"scanned_receipt.pdf text layer: {len(receipt_text)!r} (genuinely empty - a real image-only page)")


sample_invoice.pdf text layer: 197 chars
scanned_receipt.pdf text layer: 0 (genuinely empty - a real image-only page)


## 2. Labs 1-2: real text and table extraction

In [3]:
from pdf_extraction_demo import run_lab as run_extraction_lab, result_to_markdown as extraction_to_markdown

extraction_result = run_extraction_lab()
print(extraction_to_markdown(extraction_result))


# Labs 1-2 - real text and table extraction from a PDF

## Extracted text
```
Nimbus Cloud Storage - Invoice
Invoice #: INV-2026-0042
Date: 2026-07-09
Bill To: Acme Corp
Item Qty Unit Price
Personal Plan (monthly) 1 $6.99
Extra Storage (100GB) 2 $2.50
API Overage Fee 1 $4.00
```

## Extracted table

- ['Item', 'Qty', 'Unit Price']
- ['Personal Plan (monthly)', '1', '$6.99']
- ['Extra Storage (100GB)', '2', '$2.50']
- ['API Overage Fee', '1', '$4.00']

- Word-level layout entries: 33
- Rendered page image size: (1240, 1755)




## 3. Memory cost of images — real formula, real numbers

In [4]:
from local_ai_core.multimodal.memory_cost import estimate_context_budget_impact, estimate_image_tokens

for width, height in [(224, 224), (1024, 1024), (2048, 2048)]:
    tokens = estimate_image_tokens(width, height)
    fraction = estimate_context_budget_impact(tokens, context_window=8000)
    print(f"{width}x{height} image -> {tokens} tokens ({fraction:.1%} of an 8000-token context window)")


224x224 image -> 256 tokens (3.2% of an 8000-token context window)
1024x1024 image -> 5476 tokens (68.5% of an 8000-token context window)
2048x2048 image -> 21609 tokens (270.1% of an 8000-token context window)


## 4. Labs 3-5: screenshot Q&A, OCR+LLM vs VLM, full routing pipeline

`should_use_vlm()` is the "recommended pipeline principle" diagram made real: a document with a
usable text layer never reaches the VLM at all.

In [5]:
from vlm_routing_demo import run_lab as run_routing_lab, result_to_markdown as routing_to_markdown

routing_result = await run_routing_lab()
print(routing_to_markdown(routing_result))


# Labs 3-5 - screenshot Q&A, OCR+LLM vs VLM, full routing pipeline

## Lab 3: screenshot question answering (VLM-backed)
The receipt shows a Personal Plan charge of $6.99.

## Labs 4-5: automatic routing per document
- sample_invoice.pdf: routed to **text_llm** (text layer has 197 chars (>= 40 threshold) - a VLM is unnecessary)
  -> The invoice total is $6.99 for the Personal Plan.
- scanned_receipt.pdf: routed to **vlm** (text layer has only 0 chars (< 40 threshold) - likely scanned/image-only)
  -> The receipt shows a Personal Plan charge of $6.99.



## 5. Lab 6: page citations through Module 11's RAG pipeline, unchanged

In [6]:
from multimodal_rag_demo import run_lab as run_rag_lab, result_to_markdown as rag_to_markdown

rag_result = await run_rag_lab()
print(rag_to_markdown(rag_result))


# Lab 6 - page citations flowing through Module 11's RAG pipeline unchanged

- PDF-derived documents loaded: ['sample_invoice::page1', 'scanned_receipt::page1']
- Chunks ingested: ['sample_invoice::page1::0']
- Question: What is the invoice number?
- Answer: The invoice number is INV-2026-0042 [sample_invoice::page1::0].
- Chunk-level citations: ['sample_invoice::page1::0']
- Source-level (page) citations: ['sample_invoice::page1']
- Citations grounded in retrieved chunks: True

